# 03 - Extraction par expressions régulières et labellisation

Objectif : construire le label final `Urgent` / `Normal` à partir d'une règle **stricte** combinant la note et la détection de mots-clés d'urgence par regex, puis sauvegarder le dataset labellisé pour le notebook 04 (Naive Bayes).

**Règle de labellisation :**
- `Urgent` = note ≤ 3 **ET** au moins un mot-clé regex trouvé dans `review_clean`
- `Normal` = note ≥ 7 **ET** aucun mot-clé regex trouvé
- `Exclus` = tout le reste (notes 4-6, ou contradiction note/regex — ex. note ≤ 3 sans mot-clé, ou note ≥ 7 avec mot-clé)

In [1]:
import pandas as pd
import re

pd.set_option("display.max_colwidth", 120)

In [2]:
df = pd.read_csv("../data/drugs_preprocessed.csv")
df.shape

(215063, 10)

## 1. Liste de mots-clés d'urgence

Motifs couvrant les situations médicales urgentes évoquées dans les reviews, regroupés par catégorie : prise en charge d'urgence (ER, hospitalisation, 911), intoxication/idées suicidaires, réactions allergiques sévères, détresse respiratoire/cardiaque/neurologique, hémorragies/défaillance d'organe, et formulations générales de gravité immédiate (*life-threatening*, *almost died*). Une première version plus restreinte (hospitalisation/réactions graves uniquement) ne détectait que ~2,4% d'urgents ; cette liste élargie couvre davantage de situations cliniquement urgentes tout en évitant les termes génériques de mécontentement (*horrible*, *terrible*, *worst*) qui ne relèvent pas d'une urgence médicale.

Appliqués sur `review_clean` (texte non lemmatisé, pour préserver des tournures comme *can't breathe* ou *passed out*).

In [3]:
URGENT_PATTERNS = [
    # Structures d'urgence / prise en charge
    r"emergency room", r"\ber\b", r"\b911\b", r"ambulance", r"\bicu\b",
    r"urgent care", r"walk-?in clinic", r"poison control",
    r"hospital(?:ized|isation|ization)?", r"admitted to (?:the )?hospital",
    r"taken to (?:the )?(?:er|hospital|emergency)", r"rushed? to (?:the )?(?:er|hospital|emergency)",
    r"went to the er", r"called 911",
    # Intoxication / overdose / idées suicidaires
    r"overdose\w*", r"suicidal\w*", r"suicide\w*", r"self[\s-]?harm",
    r"toxic\w* reaction", r"poison\w*",
    # Réactions allergiques sévères
    r"anaphyla\w*", r"allergic reaction\w*", r"severe reaction\w*",
    r"hives\w*", r"swoll?en (?:face|throat|tongue|lips)", r"throat (?:closing|tighten\w*|swelling)",
    r"difficulty swallowing", r"steven[s]? johnson", r"rash all over",
    # Détresse respiratoire / cardiaque / neurologique
    r"can'?t breathe", r"stopped breathing", r"difficulty breathing", r"trouble breathing",
    r"shortness of breath", r"gasping for air", r"chest pain\w*", r"chest tightness",
    r"heart attack\w*", r"heart racing", r"racing heart", r"heart palpitations", r"irregular heartbeat",
    r"passed out", r"fainte?d", r"black(?:ed)? out", r"collapse\w*", r"lost consciousness", r"unconscious\w*",
    r"went into shock", r"anaphylactic shock", r"epi[\s-]?pen",
    r"seizure\w*", r"stroke\w*", r"slurred speech", r"numbness on one side", r"vision loss",
    r"throat (?:swelling shut|closing up)", r"swelling shut", r"trouble swallowing",
    # Hémorragies / organes
    r"vomiting blood", r"throwing up blood", r"blood in (?:my )?(?:stool|urine)",
    r"excessive bleeding", r"blood clot\w*", r"liver failure", r"kidney failure", r"internal bleeding",
    # Formulations générales de gravité immédiate
    r"life[\s-]?threatening", r"near death", r"almost died", r"could have died", r"thought i was going to die",
]

URGENT_REGEX = re.compile(r"\b(?:" + "|".join(URGENT_PATTERNS) + r")\b", flags=re.IGNORECASE)
print(f"{len(URGENT_PATTERNS)} motifs compilés")

72 motifs compilés


In [4]:
df["review_clean"] = df["review_clean"].astype(str)
df["regex_match"] = df["review_clean"].str.contains(URGENT_REGEX)
df["regex_match"].value_counts(normalize=True).mul(100).round(2)

regex_match
False    90.7
True      9.3
Name: proportion, dtype: float64

In [5]:
# Vérification qualitative : quelques exemples de reviews détectées comme contenant un mot-clé d'urgence
pd.set_option("display.max_colwidth", 200)
df.loc[df["regex_match"], ["rating", "review_clean"]].sample(5, random_state=42)

,rating,review_clean
115526,6,"I have been on this for 3 days now and on the first day I got severe sharp head aches in the back of my head and didn't feel as hungry as normal, second day I wasn't hungry at all and felt a littl..."
53638,10,I am a cancer patient and take strong pain meds. One night I had accidently taken other sensitive type meds and possibly second dose of narcotic pain meds. My daughter called an ambulance when she...
35707,1,"by end of day one (2 doses) I felt very tired. Day two I woke up weak, fatigued, achey but was advised to stay on Macrobid as the UTI symptoms had improved. As day wore on (3 doses) I became incre..."
33295,10,"Sustained a traumatic brain injury (TBI) and subsequently had a stroke. Depressed as to sudden limitations of what I could do after the TBI. Was on anti-depressant, CYMBALTA that had no effect. Wi..."
192493,8,I'm 3.5 months pregnant and had been suffering from the worst migraine I'd ever experienced for 2 days. I finally went to urgent care as instructed by my OB. They gave me an injection of imitrex. ...


## 2. Application de la règle de labellisation stricte

In [6]:
def label_row(row):
    if row["rating"] <= 3 and row["regex_match"]:
        return "Urgent"
    if row["rating"] >= 7 and not row["regex_match"]:
        return "Normal"
    return "Exclus"

df["label"] = df.apply(label_row, axis=1)
df["label"].value_counts()

label
Normal    131818
Exclus     76024
Urgent      7221
Name: count, dtype: int64

## 3. Rapport de distribution

In [7]:
counts = df["label"].value_counts()
pct = df["label"].value_counts(normalize=True).mul(100).round(2)

report = pd.DataFrame({"count": counts, "pct": pct})
report

,count,pct
label,,
Normal,131818,61.29
Exclus,76024,35.35
Urgent,7221,3.36


In [8]:
n_total = len(df)
n_urgent = int(counts.get("Urgent", 0))
n_normal = int(counts.get("Normal", 0))
n_exclus = int(counts.get("Exclus", 0))

print(f"Total               : {n_total:,}")
print(f"Urgent              : {n_urgent:,} ({n_urgent/n_total*100:.2f}%)")
print(f"Normal              : {n_normal:,} ({n_normal/n_total*100:.2f}%)")
print(f"Exclus              : {n_exclus:,} ({n_exclus/n_total*100:.2f}%)")
print()
n_labeled = n_urgent + n_normal
print(f"Part d'Urgent parmi les lignes labellisées (Urgent+Normal) : {n_urgent/n_labeled*100:.2f}%")

Total               : 215,063
Urgent              : 7,221 (3.36%)
Normal              : 131,818 (61.29%)
Exclus              : 76,024 (35.35%)

Part d'Urgent parmi les lignes labellisées (Urgent+Normal) : 5.19%


## 4. Sauvegarde du dataset labellisé

On ne garde que les lignes `Urgent` / `Normal` (les `Exclus` ne sont pas des exemples d'entraînement fiables pour le notebook 04), avec les colonnes `review_clean`, `label`, `rating`.

In [9]:
df_labeled = df.loc[df["label"] != "Exclus", ["review_clean", "label", "rating"]].reset_index(drop=True)
df_labeled.to_csv("../data/drugs_labeled.csv", index=False)
print("Sauvegardé :", df_labeled.shape)
df_labeled["label"].value_counts()

Sauvegardé : (139039, 3)


label
Normal    131818
Urgent      7221
Name: count, dtype: int64

## Résumé

- Règle stricte appliquée : `Urgent` = note ≤ 3 + mot-clé regex, `Normal` = note ≥ 7 + aucun mot-clé, le reste est exclu.
- Distribution rapportée ci-dessus (voir section 3) — chiffre exact à citer dans le rapport final.
- `data/drugs_labeled.csv` sauvegardé avec `review_clean`, `label`, `rating`, prêt pour le notebook 04 (Naive Bayes).